# 05 Alignment：RLHF 与 DPO——对齐与偏好优化

> 前置：`04-fine-tuning-lora`（微调）、`03-probability/07-sampling-monte-carlo`。
> 目标：理解"预训练让模型会说话，对齐让模型说人话"；拆解 RLHF 四阶段；推导并实现 **DPO**（Direct Preference Optimization），用极小例子验证其机理。

## 为什么需要对齐

预训练目标（next-token）不等于"有用、诚实、无害"。模型可能：编造事实（幻觉）、输出有害内容、拒绝帮助。**对齐（alignment）**用人类偏好把模型拉向期望行为。

## RLHF 四阶段（经典管线）

1. **SFT**：用人工标注的示范数据微调预训练模型（"学会对话形式"）；
2. **奖励模型 RM**：收集偏好对（同一回答的两个版本，人标注哪个更好），训练 RM 打分；
3. **PPO 强化学习**：以 RM 为奖励，用 PPO 优化策略——但约束它不要偏离 SFT 太远（KL 惩罚）；
4. **部署 + 迭代**：继续收集反馈，循环。

> RLHF 的痛点：要训 RM、要跑 PPO（价值网络 + 4 个模型常驻显存），工程极重。

## DPO：把 RLHF 变成分类问题

DPO（Rafailov, 2023）的关键推导：**偏好优化存在闭式最优策略**，把 RLHF 的目标函数消元，得到**只用偏好对 + 参考模型**的损失：

$$\mathcal{L}_{DPO}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\left(\beta\left[\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right]\right)\right]$$

- $y_w$：被偏好的回答；$y_l$：被拒绝的回答；
- $\pi_{ref}$：固定的参考模型（通常是 SFT 模型）；
- $\beta$：温度/强度——越大越激进地拉开好坏差距；
- 直觉：让**被偏好的回答相对参考模型升概率，被拒绝的相对降概率**。

**不再需要奖励模型和 PPO**——一阶优化即可。这是 2023 年以来开源对齐的主流（ChatGPT 仍用 RLHF，Llama/Qwen 系多用 DPO 及其变体）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 极小机理演示

3 个候选回答，动作 0 被偏好、动作 1 被拒绝。策略为可训练的 3 维 logits，参考策略均匀。直接最小化 DPO 损失，观察偏好概率变化。

In [ ]:
torch.manual_seed(0)
logits = nn.Parameter(torch.zeros(3))            # 策略（可训练）
ref_logits = torch.zeros(3)                      # 参考策略（固定，均匀）
beta = 1.0
a_w, a_l = 0, 1                                   # 偏好 0，拒绝 1

def logp(lg, a):
    return torch.log_softmax(lg, 0)[a]

def dpo_loss(lg, ref, aw, al, beta):
    inner = beta * ((logp(lg, aw) - logp(ref, aw)) - (logp(lg, al) - logp(ref, al)))
    return -torch.log(torch.sigmoid(inner) + 1e-8)

def probs(lg):
    return torch.softmax(lg, 0).detach().numpy()

print("初始策略分布:", probs(logits))
opt = torch.optim.Adam([logits], lr=0.5)
for step in range(200):
    opt.zero_grad()
    loss = dpo_loss(logits, ref_logits, a_w, a_l, beta)
    loss.backward(); opt.step()
    if step in (0, 10, 50, 199):
        print(f"step {step:3d}  loss={loss.item():.4f}  P(偏好0)={probs(logits)[0]:.3f}  P(拒绝1)={probs(logits)[1]:.3f}")

## 关键观察

1. **不训练奖励模型、不跑 PPO**：一个 `Adam` 就把偏好灌进去了——DPO 的工程红利。
2. **β 的作用**：β 小 → 温和拉开；β 大 → 悬殊（见课后练习）。β 太大容易"过拟合偏好"（过度拒绝）。
3. **局限**：DPO 是离线优化（用固定偏好数据），不迭代采集新数据；RLHF 可以在线探索。后续的 IPO/SimPO/KTO 都是 DPO 家族的改进。

In [ ]:
# β 敏感性：分别用 β=0.1 / 1.0 / 5.0 跑 100 步，看偏好概率
for beta_test in [0.1, 1.0, 5.0]:
    lg = nn.Parameter(torch.zeros(3))
    opt2 = torch.optim.Adam([lg], lr=0.5)
    for _ in range(100):
        opt2.zero_grad()
        loss = dpo_loss(lg, ref_logits, a_w, a_l, beta_test)
        loss.backward(); opt2.step()
    p = torch.softmax(lg, 0).detach().numpy()
    print(f"β={beta_test:>4.1f}  →  P(偏好0)={p[0]:.3f}  P(拒绝1)={p[1]:.3f}  P(其他2)={p[2]:.3f}")

## 总结：对齐的完整图景

```
预训练(会说话) → SFT(会对话) → RLHF/DPO(说人话) → 部署 → 反馈 → 迭代
                          ↑ 安全训练(RLAIF/红队) ↑
```

- **RLHF**：在线 + 重工程（RM + PPO）；ChatGPT 系。
- **DPO**：离线 + 轻工程；开源系（Llama 3、Qwen 均采用其变体）。
- **共同点**：核心都是"偏好对的相对概率"——本课的 DPO 损失就是全部机理。

## 课后练习

1. 把初始 logits 改成偏向拒绝项（如 `logits=[0, 2, 0]`），观察 DPO 是否能把偏好翻过来——体会"相对参考模型的上升/下降"。
2. 自己实现 DPO 的梯度：验证 `∂L/∂logits[a_w] < 0`（偏好项概率上升）且 `∂L/∂logits[a_l] > 0`。
3. 阅读：RLHF 的 PPO 目标含 KL 惩罚项，写出它与 DPO 闭式解的对应关系（提示：把 PPO 目标对策略求最优解）。